In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Data + computation
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

'''
import dask.dataframe as dd  # handle big data (optional) → Week 7–8 

# Plots
import matplotlib.pyplot as plt
import seaborn as sns # basic plots (distributions, boxplots) → Weeks 1–2
import plotly.express as px # interactive charts (time series, EDA) → Weeks 4–5
import missingno as msno # visualize missing data → Week 1

# Profiling
from ydata_profiling import ProfileReport # auto EDA report → Week 1

# Encoding + ML
import category_encoders as ce # target/frequency encoding → Week 3
import lightgbm as lgb # main baseline/strong model → Weeks 0–6
import shap # explain model predictions → Week 6

# Data validation (pick one)
!pip install pandera
import pandera as pa
# or
#import great_expectations as ge
'''

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [10]:
# Load data
train_id = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
train_trans = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train = train_trans.merge(train_id, on = 'TransactionID', how = 'left')

In [11]:
# Basic cleaning

# Fill NaN in numeric and categorical columns with median and 'missing' respectively
num_columns = train.select_dtypes(include = 'number').columns
# print(num_columns)
train[num_columns] = train[num_columns].fillna(train[num_columns].median())
categorical_cols = train.select_dtypes(include = 'object').columns
# print(categorical_cols)
train[categorical_cols] = train[categorical_cols].fillna('missing')
# train.head()

# Ordinal label encoding
# print(train.dtypes)
label_encoding = train.select_dtypes(include = 'object').columns
for col in label_encoding: 
    train[col] = train[col].astype('category').cat.codes
# train.head()

# Check correlation with the target to inform feature decison 
# correlation = train.corr()['isFraud'].sort_values(ascending=False)
# pd.set_option('display.max_rows', None)
# print(correlation)
# pd.reset_option('display.max_rows')

# Define the features and the target
features = ['TransactionAmt', 'card1', 'card3', 'card5', 'card6', 'C1', 'C2', 'C8', 'C10', 
            'C11', 'C12', 'D1', 'D2', 'D4', 'D10', 'D15', 'M1', 'M2', 'M3', 'M6', 'M7', 'M8', 'M9',  
            'V257', 'V246', 'V244', 'V242', 'V201', 'V45', 'V86', 'V87']
# exluded'ProductCD' 'DeviceType'

X = train[features]
y = train['isFraud']
print(features)

['TransactionAmt', 'card1', 'card3', 'card5', 'card6', 'C1', 'C2', 'C8', 'C10', 'C11', 'C12', 'D1', 'D2', 'D4', 'D10', 'D15', 'M1', 'M2', 'M3', 'M6', 'M7', 'M8', 'M9', 'V257', 'V246', 'V244', 'V242', 'V201', 'V45', 'V86', 'V87']


In [ ]:
# Split into train, test and validation sets 
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.2, stratify = y_temp, random_state = 42)
# print(X_train.shape) to confirm if the split worked
print(y_train.value_counts(normalize = True)) # check class proportions to confirm that stratify worked
print(y_test.value_counts(normalize = True))
print(y_val.value_counts(normalize = True))

In [ ]:
# Baseline Model - LightGBM
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
model = lgb.LGBMClassifier(n_estimators = 100, max_depth = 5, random_state = 5)
model.fit(X_train, y_train)

# Predict & evaluate
preds = model.predict_proba(X_val)[:,1]
auc = roc_auc_score(y_val, preds)
print("Baseline AUC:", auc)

In [14]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import numpy as np

# Example data
X = train[features]  # your feature columns
y = train['isFraud']

# Stratified K-Folds (preserves class balance in each fold)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = []

for train_index, val_index in kf.split(X, y):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict_proba(X_val)[:,1]
    auc = roc_auc_score(y_val, preds)
    auc_scores.append(auc)

print("CV AUC scores:", auc_scores)
print("Mean CV AUC:", np.mean(auc_scores))

[LightGBM] [Info] Number of positive: 16531, number of negative: 455901
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.057263 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3675
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034991 -> initscore=-3.317038
[LightGBM] [Info] Start training from score -3.317038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

In [15]:
_ = '''# Install pandas-profiling if you don't have it
!pip install pydantic==1.10

from pandas_profiling import ProfileReport

# Generate the report
profile = ProfileReport(train, title='Pandas Profiling Report', explorative=True)

# To display the report
profile.to_notebook_iframe()'''

hello its a me mario testing markdowns in github

# BIG HEADERRRRR

In [ ]:
# 1. IMPORT LIBRARIES
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# 2. LOAD DATA
train_id = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
train_trans = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train = train_trans.merge(train_id, on = 'TransactionID', how = 'left')

# 3. DATA CLEANING
# 3a. Fill NaN in numeric and categorical columns with median and 'missing' respectively
num_columns = train.select_dtypes(include = 'number').columns
train[num_columns] = train[num_columns].fillna(train[num_columns].median())
categorical_cols = train.select_dtypes(include = 'object').columns
train[categorical_cols] = train[categorical_cols].fillna('missing')

# 3b. Encode categorical labels (Ordinal encoding) 
label_encoding = train.select_dtypes(include = 'object').columns
for col in label_encoding: 
    train[col] = train[col].astype('category').cat.codes

# 4. FEATURE SELECTION
# 4a. Check correlation with the target to inform feature decison 
correlation = train.corr()['isFraud'].sort_values(ascending=False)
# pd.set_option('display.max_rows', None)
# print(correlation)
# pd.reset_option('display.max_rows')

# 4b. Define the features and the target
features = ['TransactionAmt', 'card1', 'card3', 'card5', 'card6', 'C1', 'C2', 'C8', 'C10', 
            'C11', 'C12', 'D1', 'D2', 'D4', 'D10', 'D15', 'M1', 'M2', 'M3', 'M6', 'M7', 'M8', 'M9',  
            'V257', 'V246', 'V244', 'V242', 'V201', 'V45', 'V86', 'V87']
# exluded'ProductCD' 'DeviceType'

X = train[features]
y = train['isFraud']

# 5. SPLIT INTO TRAIN, TEST AND VALIDATION SETS
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.2, stratify = y_temp, random_state = 42)
# Check class proportions to confirm that stratify worked
print(y_train.value_counts(normalize = True)) 
print(y_test.value_counts(normalize = True))
print(y_val.value_counts(normalize = True))

# 6. BASELINE MODEL - LIGHTGBM
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
model = lgb.LGBMClassifier(n_estimators = 100, max_depth = 5, random_state = 5)
model.fit(X_train, y_train)

# 7. PREDICT AND EVALUATE 
preds = model.predict_proba(X_val)[:,1]
auc = roc_auc_score(y_val, preds)
print("Baseline AUC:", auc)